# LoRA-tune a small model for AutoExpert (Colab)

**Before you start**
1. Runtime > Change runtime type > **T4 GPU**
2. Upload your `train.jsonl` (or `train_sample.jsonl`, renamed) using the Files panel on the left.

Each line must look like: `{"messages": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}]}`

The sample file only has 8 lines, enough to see the pipeline work. For a real difference, write **50-200 great examples** in the style you want.

In [ ]:
!pip install -q -U transformers peft trl datasets accelerate

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # small, free to download, fits a free T4
SYSTEM_PROMPT = "You are AutoExpert, a friendly expert. Give clear, short answers."

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL).to("cuda")

ds = load_dataset("json", data_files="train.jsonl", split="train")
ds = ds.train_test_split(test_size=0.1, seed=42)   # 90% train, 10% check
print(ds)

In [ ]:
def ask(m, question):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}]
    ids = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(m.device)
    out = m.generate(**ids, max_new_tokens=120, do_sample=False)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

TEST_QUESTIONS = ["What is a vector database?", "What is Git?"]
print("--- BEFORE fine-tuning ---")
for q in TEST_QUESTIONS:
    print(q, "->", ask(model, q), "\n")

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

lora = LoraConfig(
    r=16,                         # size of the "sticky notes" (bigger = more capacity)
    lora_alpha=32,                # how loudly the notes speak
    lora_dropout=0.05,
    target_modules="all-linear",  # attach notes to all the linear layers
    task_type="CAUSAL_LM",
)

args = SFTConfig(
    output_dir="out",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="no",
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    peft_config=lora,
    processing_class=tokenizer,
)
trainer.model.print_trainable_parameters()   # tiny % = the magic of LoRA
trainer.train()

In [ ]:
print("--- AFTER fine-tuning ---")
for q in TEST_QUESTIONS:
    print(q, "->", ask(trainer.model, q), "\n")

In [ ]:
# Save ONLY the small adapter (megabytes, not gigabytes) and download it
trainer.model.save_pretrained("autoexpert-lora")
tokenizer.save_pretrained("autoexpert-lora")

!zip -r autoexpert-lora.zip autoexpert-lora
from google.colab import files
files.download("autoexpert-lora.zip")

## What next?
- Run the **same 10 prompts** from `benchmark.py` on the tuned model and add it as a 3rd contestant.
- To use it in Ollama: merge the adapter into the base model, convert to GGUF with llama.cpp, then create an Ollama model from it.